# 01 – Veri Temizliği ve Keşifsel Veri Analizi (EDA)
**Hafta:** H3–H4 (02.11 – 15.11) · **Puanlama:** EDA ve ön işleme %20 · **Slaytlar:** 20–30

**Amaç:** Veriyi tanımak; kalite sorunlarını, dengesizliği ve modellemeyi etkileyecek özellikleri bulmak.
Bu notebook'ta **model eğitilmez**. Temizlik kararı en sonda verilir.

Her adım: başlık → sözde kod → küçük kod hücresi → çıktı → 1–2 cümle yorum.

### Adım 0: Kurulum
*Slayt 55*

**Ne yapıyoruz:** Kütüphaneleri ve proje kodunu yüklüyoruz.

In [1]:
# proje kökünü Python yoluna ekle (src/ klasörünü kullanabilmek için)
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# proje kodu
from src import config
from src.data import load_raw, clean

pd.set_option("display.max_columns", 50)

### Adım 1: Veriyi yükle ve ilk bakış
*Slayt 15–16, 20*

**Ne yapıyoruz:** Veriyi okuyup kaç satır (örnek) ve kaç sütun (değişken) olduğuna bakıyoruz.

**Sözde kod (slayt 57):**
```
veri = dosyayı oku
satır ve sütun sayısını yazdır
ilk 5 satırı göster
```

In [2]:
# veriyi yükle
raw = load_raw()

# boyut: (satır, sütun)
print("Boyut:", raw.shape)

# ilk satırlar
raw.head()

Boyut: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


**Yorum:** (Her satır bir oturum, her sütun bir değişken. Hedef sütun hangisi?)

### Adım 2: Değişken tipleri
*Slayt 21, 24*

**Ne yapıyoruz:** Hangi sütunlar sayısal, hangileri kategorik? Sayıyla kodlanmış ama aslında kategori olanlar var mı?

**Sözde kod (slayt 57):**
```
her sütunun veri tipini yazdır
sayıyla kodlanmış kategorikleri işaretle
```

In [3]:
# sütunların veri tipleri
raw.dtypes

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

**Yorum:** 

### Adım 3: Veri kalitesi: eksik değer ve tekrar eden satır
*Slayt 22–23*

**Ne yapıyoruz:** Eksik değer, birebir tekrar eden satır ve mantıksız değer (ör. negatif süre) var mı?

**Sözde kod (slayt 57):**
```
her sütundaki eksik değer sayısını say
tekrar eden satır sayısını say
tekrar eden satırlar nasıl satırlar? incele
sayısal sütunlarda negatif / aralık dışı değer var mı?
Month yazımlarını listele
```

**Kontrol soruları:**
- Tekrar eden satırlar bölmeden önce mi silinmeli, sonra mı? Neden? (slayt 35, 37)
- `Month` sütunundaki değerlerin yazımı tutarlı mı?

In [4]:
# eksik değerler: boş hücre varsa modelden önce doldurmak ya da silmek gerekir
missing = raw.isna().sum()
print("Toplam eksik değer:", missing.sum())

# tekrar eden satırlar: birebir aynı satırlar bölmede eğitim ve teste dağılabilir
dup_mask = raw.duplicated()
print("Tekrar eden satır:", dup_mask.sum())

Toplam eksik değer: 0
Tekrar eden satır: 125


In [5]:
# tekrar eden satırlar nasıl satırlar? veri hatası mı, gerçekten aynı görünen oturumlar mı?
dups = raw[dup_mask]
print("Tekrarlarda alışveriş oranı:", round(dups[config.TARGET].mean(), 3))

# birkaç tipik sütunun medyanı: bu oturumlar ne kadar "dolu"?
look_cols = ["ProductRelated", "ProductRelated_Duration", "PageValues", "BounceRates", "ExitRates"]
dups[look_cols].median()

Tekrarlarda alışveriş oranı: 0.0


ProductRelated             1.0
ProductRelated_Duration    0.0
PageValues                 0.0
BounceRates                0.2
ExitRates                  0.2
dtype: float64

In [6]:
# mantıksız değerler: sayfa sayısı ve süre negatif olamaz
negatives = (raw[config.NUM_COLS] < 0).sum()
print("Negatif değer sayısı:", negatives.sum())

# oranlar ve SpecialDay tanım gereği 0 ile 1 arasında olmalı
rate_cols = ["BounceRates", "ExitRates", "SpecialDay"]
raw[rate_cols].agg(["min", "max"])

Negatif değer sayısı: 0


,BounceRates,ExitRates,SpecialDay
min,0.0,0.0,0.0
max,0.2,0.2,1.0


In [7]:
# Month yazımları: zaman bölmesi ay adlarıyla yapılacağı için yazım tutarlı olmalı
raw["Month"].value_counts()

Month
May     3364
Nov     2998
Mar     1907
Dec     1727
Oct      549
Sep      448
Aug      433
Jul      432
June     288
Feb      184
Name: count, dtype: int64

**Yorum:** Eksik ve mantıksız değer yok. 125 tekrarın hepsi tek ürün sayfasında 0 saniye kalıp çıkan, alışverişsiz oturumlar; bölmeden önce silinecekler (Adım 10), yoksa aynı satır hem eğitimde hem testte olup skoru şişirebilir. `Month`'ta Haziran `June` yazılmış, Ocak ve Nisan hiç yok; zaman bölmesi `config.MONTH_ORDER`'daki bu yazımı kullanıyor.

### Adım 4: Hedef değişken ve dengesizlik
*Slayt 27, 46*

**Ne yapıyoruz:** `Revenue` sınıflarının dağılımına bakıyoruz.

**Sözde kod (slayt 57):**
```
Revenue değerlerini say (adet ve oran)
çubuk grafik çiz
herkese 'almaz' deseydik accuracy kaç olurdu?
```

**Kontrol soruları:**
- Bu dağılım metrik seçimini nasıl etkiliyor?

In [8]:
# sınıf sayıları ve oranları

# çubuk grafik (reports/figures/01_hedef_dagilimi.png olarak kaydedin)

**Yorum:** 

### Adım 5: Sayısal değişkenler: dağılım, çarpıklık, aykırı değer
*Slayt 25–26*

**Ne yapıyoruz:** Sayısal değişkenlerin özet istatistiklerine ve dağılımlarına bakıyoruz.

**Sözde kod (slayt 57):**
```
describe() ile özet istatistik
süre değişkenlerinin histogramı
aykırı değerler hata mı, gerçek davranış mı? karar ver
```

**Kontrol soruları:**
- Hangi değişkenler çok çarpık? Log dönüşümü gerekir mi?
- Hangi değişkenlerin ölçeği çok farklı? Bu hangi modelleri etkiler? (slayt 25)

In [9]:
# özet istatistikler

# histogramlar

**Yorum:** 

### Adım 6: Kategorik değişkenler
*Slayt 24*

**Ne yapıyoruz:** Kategori sayıları ve sıklıkları. Sayıyla kodlanmış kategoriklerde sayıların sırası anlam taşıyor mu?

**Sözde kod (slayt 57):**
```
her kategorik sütun için benzersiz değer sayısı
en sık kategoriler
çok seyrek kategoriler var mı?
```

In [10]:
# kategorik sütunlar: config.CAT_COLS

**Yorum:** 

### Adım 7: Zaman boyutu: aylar
*Slayt 35*

**Ne yapıyoruz:** Zamana göre bölme deneyi için ayların dağılımına bakıyoruz.

**Sözde kod (slayt 57):**
```
aylara göre oturum sayısı ve alışveriş oranı (config.MONTH_ORDER sırasıyla)
hangi aylar eksik?
SpecialDay hangi aylarda sıfırdan farklı?
```

**Kontrol soruları:**
- Alışveriş oranı aylara göre değişiyorsa zamana göre bölmede ne olur?

In [11]:
# aylara göre oturum sayısı ve alışveriş oranı

**Yorum:** 

### Adım 8: PageValues ve alışveriş ilişkisi
*Slayt 37–38*

**Ne yapıyoruz:** Sızıntı deneyinin gerekçesi: PageValues alışverişle ne kadar ilişkili?

**Sözde kod (slayt 57):**
```
PageValues = 0 olan oturumların oranı
PageValues > 0 ve = 0 gruplarında alışveriş oranı
alıcıların kaçında PageValues = 0?
```

**Kontrol soruları:**
- Google Analytics'in Page Value tanımına göre bu değer tahmin anında biliniyor mu?

In [12]:
# PageValues grupları ve alışveriş oranları

**Yorum:** 

### Adım 9: Diğer ilişkiler
*Slayt 28*

**Ne yapıyoruz:** `VisitorType`, `Weekend`, `BounceRates`/`ExitRates` ile alışveriş ilişkisi; sayısal değişkenler arası korelasyon.

**Sözde kod (slayt 57):**
```
ziyaretçi tipine göre alışveriş oranı
korelasyon ısı haritası
```

In [13]:
# ilişkiler

**Yorum:** 

### Adım 10: Temizlik kararı
*Slayt 22, 35*

**Ne yapıyoruz:** EDA bulgularına göre temizliği uyguluyoruz: `clean()` tekrar eden satırları siler ve True/False'u 0/1 yapar.

**Sözde kod (slayt 57):**
```
df = clean(raw)
satır sayısı ve alışveriş oranı tekrar kontrol
```

In [14]:
# temizliği uygula
df = clean(raw)

# kontrol
print("Satır:", len(df))
print("Alışveriş oranı:", round(df[config.TARGET].mean(), 3))

Tekrar eden satır silindi: 125  (kalan: 12205)
Satır: 12205
Alışveriş oranı: 0.156


**Yorum:** 

## Bulgular ve kararlar
> Bu hücreyi **kendi cümlelerinizle** yazın. Sunumda ve hocanın kontrolünde bunu savunacaksınız.

- Bulgu 1:
- Bulgu 2:
- Bu bulgulara göre aldığımız karar(lar):
- Deney günlüğüne yazıldı mı? (`reports/deney_gunlugu.md`)
- Git commit atıldı mı?
- `data/README.md` → "Bilinen sınırlamalar" bölümü güncellendi mi?